# Pattern-Matching NILM

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat

## Directories

In [8]:
base_dir = os.getcwd()
processed_data_dir = os.path.join(base_dir, 'processed_data')

results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

## Load Data

In [9]:
processed_data_filepath = os.path.join(processed_data_dir, 'ampds_dataset.npz')
data = np.load(processed_data_filepath)
print(data.files)

['x_norm', 'y_norm', 'x_min', 'x_max', 'y_min', 'y_max', 'train_inp', 'train_out', 'val_inp', 'val_out', 'test_inp', 'test_out', 'T_limit', 'train_val_test_split', 'window_length', 'stride']


## Create Template Database

Pattern-matching algorithms work by comparing new inputs with examples stored from the training data. Here, the normalizes data and training input/output indices already define those examples, so there's no need to build a separate template database.

## Sample Generation

To create a sample, we use the normalized aggregate and appliance profiles (`y_norm` and `x_norm`), along with the input/output indices and window length, to select an input window and its corresponding target.

In [10]:
def generate_sample(x, y, i_inp, i_out, window_length):
    x_win = x[i_inp : i_inp + window_length]
    y_target = y[i_out]
    return x_win, y_target

## Time-series data frame comparison

### Feature distance

One way to compare time-series frames is to summarize each with a feature vector, then measure the difference between those vectors. Possible features include the mean, minimum, maximum, standard deviation, root-mean-square (RMS), and energy.

In [27]:
def compute_feature_vector(x_win, feature_selection='all'):

    funcs = {
        'mean': lambda x: np.mean(x),
        'min': lambda x: np.min(x),
        'max': lambda x: np.max(x),
        'std': lambda x: np.std(x),
        'rms': lambda x: np.mean(x**2),
        'energy': lambda x: np.sum(x**2)}

    if feature_selection == 'all': feature_selection = list(funcs.keys())
    out = np.column_stack([funcs[name](x_win) for name in feature_selection])
    return out

In [12]:
def feature_distance(vector1, vector2): return np.linalg.norm(vector1 - vector2)

Recomputing template features for every test-frame comparison would be inefficient. Instead, we compute and store each template's feature vector once, then use a lookup keyed by the template's input index to retrieve it.

In [28]:
feature_lookup = {
    int(i): compute_feature_vector(data['x_norm'][i : i + data['window_length'].item()])
    for i in data['train_inp']}

In [32]:
columns = ["Mean", "Min", "Max", "Std", "RMS", "Energy"]
df = pd.DataFrame([v.ravel() for v in list(feature_lookup.values())[:5]],
                  index=list(feature_lookup)[:5], columns=columns)
df.index.name = "Template Start Index"
display(df.reset_index().style.hide(axis="index"))

Template Start Index,Mean,Min,Max,Std,RMS,Energy
22,0.144452,0.000000,0.472586,0.148209,0.042832,7.709837
706,0.193100,0.000000,0.833049,0.276813,0.113913,20.504334
218,0.057543,0.000000,0.247247,0.080427,0.009780,1.760362
335,0.190778,0.000000,0.386913,0.156774,0.060974,10.975402
664,0.220624,0.000000,0.693694,0.242901,0.107676,19.381675


### Correlation

another way to compare the two time series data frames is a correlation

In [ ]:
def compute_correlation(frame1, frame2): return np.corrcoef(frame1.flatten(), frame2.flatten())[0, 1]